# Custom Drivers: Overriding `_load`

By default, `Data._load(path)` delegates to the class's `driver` to read the
cached file and return the result. For most raster and vector formats the
built-in drivers (`RioXArrayDriver`, `GeoPandasDriver`, …) are enough.

When the default is not enough — non-standard formats, composite outputs,
derived results — you override `_load(self, path)` directly.

This notebook covers:

- The default `_load` flow and when to override it
- A complete example: saving statistics to `.npz` in `_process` and reading
  them back as a `dict` in `_load`
- Bundling `_process` and `_load` together for a self-contained loader

In [ ]:
import os, sys
os.chdir('../../..')
sys.path.insert(0, 'docs/loaders')

In [ ]:
from pygeodata import SpatialSpec, get_config, load

get_config().update(path_cache='data/processed')

spec = SpatialSpec.from_raster_file('data/wtd.tif')
print('Target spec:', spec)

## 1. The default `_load` flow

When you call `load(loader, spec)`, pygeodata:

1. Resolves the spec and ensures the output is processed
2. Calls `loader._load(path)` where `path = loader.get_processed_path(spec)`
3. The default implementation passes `path` to `loader.driver`

```python
# Default implementation in Data
def _load(self, path: Path):
    return self.driver(path)
```

To change what gets returned, override `_load` in your subclass. You still
receive the resolved `path` — you are not responsible for resolving or
processing it.

## 2. Example: saving stats to `.npz`, loading as a dict

Suppose we want to compute summary statistics (mean, std, min, max) of the
land water-table depth and cache them. NumPy's `.npz` format is a natural
choice, but there is no built-in driver that reads `.npz` and returns a `dict`.

```python
# docs/loaders/pipeline.py

@dataclass
class WTDStatsLoader(Data):
    wtd: WaterTableDepthLoader
    mask: CountryMaskLoader

    ext = 'npz'

    def _process(self, spec: SpatialSpec) -> None:
        da = load(self.wtd, spec).where(load(self.mask, spec) == 1)
        values = da.values
        path = self.get_processed_path(spec)
        path.parent.mkdir(parents=True, exist_ok=True)
        np.savez(path,
                 mean=values[~np.isnan(values)].mean(),
                 std=values[~np.isnan(values)].std(),
                 min=values[~np.isnan(values)].min(),
                 max=values[~np.isnan(values)].max())

    def _load(self, path: Path) -> dict:
        data = np.load(path)
        return {k: float(data[k]) for k in data.files}
```

Key points:

- **No `driver` needed.** When you override `_load`, the `driver` class
  attribute is never called, so you can omit it.
- **`ext` must match.** pygeodata uses `ext` to construct the cache path;
  `np.savez` writes a `.npz` file, so `ext = 'npz'`.
- **`path` is the resolved cache path.** You do not call `get_processed_path`
  inside `_load` — pygeodata already passes the right path.

In [ ]:
from pipeline import WTDStatsLoader, WaterTableDepthLoader, CountryMaskLoader

stats_loader = WTDStatsLoader(
    wtd=WaterTableDepthLoader(),
    mask=CountryMaskLoader(),
)

stats = load(stats_loader, spec)
print('Return type:', type(stats))
for k, v in stats.items():
    print(f'  {k}: {v:.3f} m')

The cache file on disk is a real `.npz` file. Calling `load()` a second time
skips processing and returns the same dict directly from the file.

In [ ]:
path = stats_loader.get_processed_path(spec)
print('Cache path:', path)
print('Exists:    ', path.exists())
print('Is valid:  ', stats_loader.is_cache_valid(spec))

# Second load — no reprocessing
stats2 = load(stats_loader, spec)
assert stats == stats2

## 3. When to override `_load`

| Situation | Approach |
|-----------|----------|
| Standard raster format (GeoTIFF, NetCDF) | Use `RioXArrayDriver` (default) |
| Standard vector format (Shapefile, GeoJSON, Parquet) | Use `GeoPandasDriver` or `GeoPandasParquetDriver` |
| Numpy `.npy` / `.npz` | Override `_load` |
| CSV / JSON / pickle | Override `_load` |
| Post-processing of loaded data (clipping, casting) | Override `_load` |
| Multiple arrays stored in a single file | Override `_load` |

The rule of thumb: if you would write a one-liner `lambda path: ...` to
describe how to read the file, override `_load`. If the format is already
handled by a built-in driver, use that instead.

## 4. Post-processing in `_load`

You can also override `_load` to apply a transformation on top of what a
driver would normally return — e.g. select a band, cast dtype, or clip values.

```python
@dataclass
class ElevationUInt16(Data):
    """Elevation reprojected and cast to uint16 on load."""

    src: str = 'data/elevation.tif'
    driver = RioXArrayDriver()

    @property
    def processor(self):
        return Reprojector(src_path=self.src)

    def _load(self, path: Path):
        da = self.driver(path)           # use the driver as a callable
        return da.clip(0).astype('uint16')
```

Calling `self.driver(path)` inside a custom `_load` is perfectly valid —
the driver is just a callable that reads a file.